<a href="https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy python-dotenv

import os
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04"]
daily_files = [
    hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                     filename=f"fact_content_daily_performance/month={m}/data_0.parquet", token=token)
    for m in MONTHS
]
dim_content_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                    filename="dim_content.parquet", token=token)
clients_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                filename="dim_clients.parquet", token=token)

con = duckdb.connect()
file_list = ", ".join(f"'{f}'" for f in daily_files)
REL = f"read_parquet([{file_list}])"
DECISION_DATE = "2026-03-31"

query = f"""
WITH prior AS (
    SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions, SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_sum_position) AS gsc_sum_position,
        -- Reproduces the discarded gate (`gsc_avg_position > 0`) so the next
        -- cell can measure what it used to throw away. Not a feature.
        SUM(gsc_impressions) FILTER (WHERE gsc_avg_position > 0) AS old_gate_impressions,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
        COUNT(*) FILTER (WHERE ga4_sessions > 0) AS days_with_sessions,
        BOOL_OR(ga4_data_available) AS ga4_data_available,
        SUM(ga4_pageviews) AS ga4_pageviews, SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_users) AS ga4_users, SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,
        SUM(sessions_organic) AS sessions_organic, SUM(sessions_direct) AS sessions_direct,
        SUM(scroll_events) AS scroll_events,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 60 DAY
              AND report_date < DATE '{DECISION_DATE}' - INTERVAL 30 DAY
        ) AS trend_baseline_impr,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 30 DAY
        ) AS trend_recent_impr
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
      AND report_date < DATE '{DECISION_DATE}'
    GROUP BY content_hash_id
),
future AS (
   
    SELECT content_hash_id, SUM(gsc_impressions) AS future_impressions
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}'
      AND report_date < DATE '{DECISION_DATE}' + INTERVAL 30 DAY
    GROUP BY content_hash_id
)
SELECT p.*, f.future_impressions
FROM prior p JOIN future f USING (content_hash_id)
WHERE p.trend_baseline_impr > 0 AND p.trend_recent_impr > 0
-- Deterministic row order. Without it DuckDB may return rows in any order, so
-- the train/test split shifts between runs even with a fixed seed -- results
-- stop being reproducible from a fresh clone.
ORDER BY content_hash_id
"""
df = con.sql(query).df()

# Google's documented formula for GSC bulk-export data (https://support.google.com/webmasters/answer/12917991):
#   average position (1-based) = SUM(sum_position) / SUM(impressions) + 1
# sum_position is ZERO-based -- 0 is the top result. No filtering is needed:
# zero-impression rows carry sum_position = 0, so they contribute nothing to
# either side (verified: 0 rows violate this).
df["gsc_avg_position"] = df["gsc_sum_position"] / df["gsc_impressions"] + 1
assert df["gsc_avg_position"].notna().all(), "cohort should guarantee prior-window impressions"

# dim_content carries its own client_hash_id; dropping it avoids a column
# collision that silently breaks the dim_clients merge below.
dim = con.sql(f"SELECT * FROM read_parquet('{dim_content_file}')").df()
dim = dim.drop(columns=["client_hash_id"])
df = df.merge(dim, on="content_hash_id", how="left")

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM read_parquet('{clients_file}')").df()
df = df.merge(clients, on="client_hash_id", how="left")

print("Rows before any filtering:", len(df))

# Row filters named in the data contract but not previously applied. Deleted
# pages decline at 86.6% vs 54.0% for live ones -- an artefact of removal, not
# an SEO signal. Few in number here, but they do not belong in the population.
live = df["is_deleted"].fillna(False).eq(False) & df["is_published"].fillna(True).eq(True)
print("Dropped as deleted/unpublished:", (~live).sum(), f"({(~live).mean():.2%})")
df = df[live].copy()

prior_window_start = pd.Timestamp(DECISION_DATE) - pd.Timedelta(days=90)
coverage_ok = df["gsc_data_start"].isna() | (df["gsc_data_start"] <= prior_window_start)
print("Dropped for incomplete client coverage:", (~coverage_ok).sum(),
      f"({(~coverage_ok).mean():.1%})")
df = df[coverage_ok].copy()
print("Rows after coverage filter:", len(df))
print()

df["prior_trend_pct"] = (df["trend_recent_impr"] - df["trend_baseline_impr"]) / df["trend_baseline_impr"] * 100
df["was_declining"] = df["prior_trend_pct"] <= -20

decision_ts = pd.Timestamp(DECISION_DATE)
df["content_age_days"] = (decision_ts - pd.to_datetime(df["content_created_date"])).dt.days
df["days_since_last_update"] = (decision_ts - pd.to_datetime(df["content_updated_date"])).dt.days

df["ctr"] = (df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan) * 100).fillna(0)
df["engagement_rate"] = (df["ga4_engaged_sessions"] / df["ga4_sessions"].replace(0, np.nan) * 100).fillna(0)
df["scroll_rate"] = (df["scroll_events"] / df["ga4_pageviews"].replace(0, np.nan) * 100).fillna(0)

# Flags must be computed BEFORE the fills below, or the missingness is erased.
df["has_ga4_data"] = df["ga4_data_available"].fillna(False).astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_backlink_data"] = df["backlinks"].notna().astype(int)

for col in ["search_volume", "competition", "cpc", "word_count", "char_count", "backlinks",
            "ga4_pageviews", "ga4_sessions", "ga4_users", "ga4_engaged_sessions",
            "ga4_total_engagement_sec", "sessions_organic", "sessions_direct", "scroll_events"]:
    df[col] = df[col].fillna(0)
df["main_intent"] = df["main_intent"].fillna("unknown")
df["content_type"] = df["content_type"].fillna("unknown")
df["competition_level"] = df["competition_level"].fillna("unknown")

for col in ["gsc_impressions", "gsc_clicks", "ga4_sessions", "search_volume", "backlinks",
            "scroll_events", "gsc_sum_position", "ga4_engaged_sessions"]:
    df[f"log_{col}"] = np.log1p(df[col])

print("Feature vector shape:", df.shape)
df.head()


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rows before any filtering: 134398
Dropped as deleted/unpublished: 150 (0.11%)
Dropped for incomplete client coverage: 18645 (13.9%)
Rows after coverage filter: 115603



Feature vector shape: (115603, 65)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,old_gate_impressions,days_with_impressions,days_with_sessions,ga4_data_available,ga4_pageviews,...,has_word_count,has_backlink_data,log_gsc_impressions,log_gsc_clicks,log_ga4_sessions,log_search_volume,log_backlinks,log_scroll_events,log_gsc_sum_position,log_ga4_engaged_sessions
0,content_000005d4ced12088,client_9958f0a7ae1df715,126.0,0.0,9233.0,126.0,49,0,False,0.0,...,0,0,4.844187,0.000000,0.0,4.70953,0.0,0.0,9.130648,0.0
1,content_00007bd2985b77c3,client_73cda7b4e4f265ea,70.0,0.0,385.0,42.0,37,0,False,0.0,...,0,0,4.262680,0.000000,0.0,2.397895,0.0,0.0,5.955837,0.0
2,content_0000cd28fbda69f3,client_3ffa76342f366962,57.0,1.0,390.0,56.0,31,0,False,0.0,...,1,0,4.060443,0.693147,0.0,0.0,0.0,0.0,5.968708,0.0
3,content_0000d495bfbfb4a8,client_2094c6eb080311d5,32.0,0.0,178.0,30.0,6,0,False,0.0,...,1,1,3.496508,0.000000,0.0,8.999743,5.828946,0.0,5.187386,0.0
4,content_00014efc121d911d,client_08a6a72ff48e62c0,130.0,1.0,793.0,125.0,42,0,<NA>,0.0,...,0,0,4.875197,0.693147,0.0,0.0,0.0,0.0,6.677083,0.0


**Experiment: what does `gsc_avg_position = 0` mean?**

*Prompted by mentor materials.* The Week 3 lecture (*Google Cloud Data Aggregation & Sync*, Haris) buries the clue in a resource link: **`sum_position` is zero-based — the official average adds +1** — "a classic sharp edge." [Google's own reference](https://support.google.com/webmasters/answer/12917991) confirms it: *"To calculate average position (which is 1-based), calculate SUM(sum_position)/SUM(impressions) + 1."* The Week 4 lecture (*Content Optimization*) supplies the behavioural test: CTR must be read against peers at a similar position, because *"0.5% CTR at position 7 is weak; the same 0.5% at position 40 is normal."* The refreshed `docs/data-dictionary.md` adds a figure to check against — at warehouse scale, positions 1–3 run **≈2.78% CTR** — and warns that tier metrics need a volume floor.

That contradicts the starter CSV's dictionary, which says `avg_position = 0` marks **missing data**. Both cannot hold for the same column, and the answer decides whether a `0` is the best possible rank or no rank at all — the difference between keeping a page's best days and deleting them.

Six checks below. Three support the zero-based reading; three test whether the resulting ranks behave like real ranks.

In [2]:
# --- Check 1: are there values below 1 among page-days that DID appear? ---
# Gate on impressions: if a page got none, it never appeared and has no rank,
# so a 0 there would be meaningless. With impressions, a rank must exist.
print("CHECK 1 -- page-days with impressions (a rank must exist)")
print(con.sql(f"""
SELECT ROUND(MIN(gsc_avg_position), 2)                                 AS min_position,
       COUNT(*) FILTER (WHERE gsc_avg_position > 0 AND gsc_avg_position < 1) AS between_0_and_1,
       COUNT(*) FILTER (WHERE gsc_avg_position = 0)                     AS exactly_zero,
       COUNT(*)                                                         AS page_days
FROM {REL} WHERE gsc_impressions > 0
""").df().to_string(index=False))
print("Real GSC ranks start at 1, so nothing should fall below it.\n")

# --- Check 2: is the column the raw ratio, or already corrected? ---
print("CHECK 2 -- does the column equal sum_position / impressions exactly?")
print(con.sql(f"""
SELECT COUNT(*) AS rows_that_differ
FROM {REL}
WHERE gsc_impressions > 0
  AND ABS(gsc_avg_position - gsc_sum_position / gsc_impressions) > 0.001
""").df().to_string(index=False))
print("0 means no +1 has been applied for us.\n")

# --- Check 3: single-impression days -- no averaging can occur ---
# With exactly one impression, sum_position IS that impression's rank.
print("CHECK 3 -- days with exactly ONE impression (sum_position = the raw rank)")
print(con.sql(f"""
SELECT gsc_sum_position AS observed_rank, COUNT(*) AS n
FROM {REL} WHERE gsc_impressions = 1
GROUP BY 1 ORDER BY observed_rank LIMIT 5
""").df().to_string(index=False))
print("A single impression cannot average to anything. Rank 0 is impossible one-based.\n")

# --- Check 4: do the supposed rank-1 pages behave like rank-1 pages? ---
print("CHECK 4 -- click behaviour of rows at sum_position = 0")
print(con.sql(f"""
SELECT COUNT(*) AS n_rows, SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
       ROUND(SUM(gsc_clicks) * 100.0 / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr_pct
FROM {REL} WHERE gsc_impressions > 0 AND gsc_sum_position = 0
""").df().to_string(index=False))
print("If these are rank-1 pages, CTR should be very high (~25% in the real world).\n")

# --- Check 5: does CTR fall with rank, as search behaviour requires? ---
print("CHECK 5 -- CTR by raw position bucket")
print(con.sql(f"""
SELECT CASE WHEN gsc_avg_position < 1  THEN 'raw 0-1   (= rank 1-2)'
            WHEN gsc_avg_position < 3  THEN 'raw 1-3   (= rank 2-4)'
            WHEN gsc_avg_position < 9  THEN 'raw 3-9   (= rank 4-10)'
            WHEN gsc_avg_position < 19 THEN 'raw 9-19  (= page 2)'
            ELSE                            'raw 19+   (= page 3+)' END AS bucket,
       COUNT(*) AS n_page_days, SUM(gsc_impressions) AS impressions,
       ROUND(SUM(gsc_clicks) * 100.0 / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr_pct
FROM {REL} WHERE gsc_impressions > 0
GROUP BY 1 ORDER BY MIN(gsc_avg_position)
""").df().to_string(index=False))
print("CTR should fall steeply as rank worsens.\n")

# --- Impact of the gate this experiment replaced ---
lost = df["gsc_impressions"] - df["old_gate_impressions"].fillna(0)
print("IMPACT of the old `gsc_avg_position > 0` gate on this cohort")
print(f"  pages that lost impressions from their position average: {(lost > 0).sum():,}"
      f" ({(lost > 0).mean():.1%})")
print(f"  impressions discarded:                                   {lost.sum():,.0f}"
      f" ({lost.sum() / df['gsc_impressions'].sum():.2%} of all)")

# --- Check 6: does a volume floor rescue the CTR curve? ---
# docs/data-dictionary.md warns that tier metrics need a volume floor, and gives
# a figure to check against: positions 1-3 run ~2.78% CTR at warehouse scale.
# Aggregate per PAGE first (impression-weighted rank), then bucket.
print("CHECK 6 -- CTR by rank, per page, at three volume floors")
for floor in (0, 100, 1000):
    out = con.sql(f"""
    WITH per_page AS (
      SELECT content_hash_id, SUM(gsc_impressions) AS impr, SUM(gsc_clicks) AS clicks,
             SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) + 1 AS rank_pos
      FROM {REL} WHERE gsc_impressions > 0 GROUP BY 1)
    SELECT CASE WHEN rank_pos < 4  THEN 'rank 1-3'  WHEN rank_pos < 11 THEN 'rank 4-10'
                WHEN rank_pos < 21 THEN 'page 2'    ELSE 'page 3+' END AS bucket,
           COUNT(*) AS pages, ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impr), 0), 2) AS ctr_pct
    FROM per_page WHERE impr >= {floor}
    GROUP BY 1 ORDER BY MIN(rank_pos)
    """).df()
    print(f"  floor >= {floor} impressions:  " +
          "  ".join(f"{r.bucket} {r.ctr_pct}%" for r in out.itertuples()))
print("  data-dictionary reference for rank 1-3 at warehouse scale: 2.78%")

CHECK 1 -- page-days with impressions (a rank must exist)


 min_position  between_0_and_1  exactly_zero  page_days
          0.0           451480        838976   14618722
Real GSC ranks start at 1, so nothing should fall below it.

CHECK 2 -- does the column equal sum_position / impressions exactly?


 rows_that_differ
                0
0 means no +1 has been applied for us.

CHECK 3 -- days with exactly ONE impression (sum_position = the raw rank)


 observed_rank      n
             0 394479
             1  50073
             2  42408
             3  53871
             4  62482
A single impression cannot average to anything. Rank 0 is impossible one-based.

CHECK 4 -- click behaviour of rows at sum_position = 0


 n_rows  impressions  clicks  ctr_pct
 838987    2875734.0  5591.0     0.19
If these are rank-1 pages, CTR should be very high (~25% in the real world).

CHECK 5 -- CTR by raw position bucket


                 bucket  n_page_days  impressions  ctr_pct
 raw 0-1   (= rank 1-2)      1290456   28996829.0     0.17
 raw 1-3   (= rank 2-4)      1677799  155884797.0     0.42
raw 3-9   (= rank 4-10)      5453341  550212491.0     0.33
   raw 9-19  (= page 2)      2712192  116469611.0     0.31
  raw 19+   (= page 3+)      3484934  158051631.0     0.14
CTR should fall steeply as rank worsens.

IMPACT of the old `gsc_avg_position > 0` gate on this cohort
  pages that lost impressions from their position average: 61,683 (53.4%)
  impressions discarded:                                   1,631,795 (0.30% of all)
CHECK 6 -- CTR by rank, per page, at three volume floors


  floor >= 0 impressions:  rank 1-3 0.4%  rank 4-10 0.32%  page 2 0.32%  page 3+ 0.15%


  floor >= 100 impressions:  rank 1-3 0.4%  rank 4-10 0.32%  page 2 0.32%  page 3+ 0.15%


  floor >= 1000 impressions:  rank 1-3 0.4%  rank 4-10 0.32%  page 2 0.32%  page 3+ 0.15%
  data-dictionary reference for rank 1-3 at warehouse scale: 2.78%


**Verdict: the format is zero-based, but the clicks do not behave like it.**

**Supporting zero-based.** Check 3 is decisive: on days with exactly one impression no averaging is possible, so `sum_position` *is* the observed rank — and rank `0` appears **394,479** times. Impossible if 1 were the floor. Check 1 agrees (451,480 page-days between 0 and 1) and check 2 shows no `+1` has been applied for us. This matches the export format the W3 lecture describes.

**Contradicting it.** If raw `0` means rank 1, those pages should earn excellent click rates. Check 4: 838,987 rows at `sum_position = 0` earn **0.19% CTR** across 2.88M impressions. Check 5 shows CTR barely varying with rank — the supposed top bucket (0.17%) is the second *worst*. Check 6 rules out the obvious explanation: the data dictionary warns that tier metrics need a volume floor, but the curve stays flat at every floor tested:

| rank bucket | floor ≥0 | ≥100 | ≥1,000 |
|---|---|---|---|
| **rank 1–3** | **0.40%** | **0.40%** | **0.40%** |
| rank 4–10 | 0.32% | 0.32% | 0.32% |
| page 2 | 0.32% | 0.32% | 0.32% |
| page 3+ | 0.15% | 0.15% | 0.15% |

`docs/data-dictionary.md` states positions 1–3 run **≈2.78%** at warehouse scale. This release gives **0.40%** — a **7x gap against FlyRank's own documented figure**, stable across volume floors.

**What that implies.** The click-to-impression relationship does not survive in this release as documented. Any CTR-derived feature is unreliable here, and so is the position-banded peer comparison W4 describes — the mechanism FlyRank's own `low_ctr_visible_page` rule depends on. Recorded as ML-06 hypothesis 5, and worth raising with the mentor: the discrepancy is against their published number, not merely against intuition.

**What this means for the fix.** Gating on impressions and adding `+1` remains correct — it follows the documented format and stops discarding 53.4% of pages' best days. But `gsc_avg_position` should not be treated as a precise rank, and CTR should not be trusted at all, until hypothesis 5 is settled.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Known before decision point? |
|---|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` | 90-day GSC totals | zero-fill | Yes |
| `gsc_avg_position` | `SUM(sum_position)/SUM(impressions) + 1` — [Google's documented formula](https://support.google.com/webmasters/answer/12917991). `sum_position` is zero-based, so the `+1` yields a real 1-based rank. Not a mean of daily means — that would let one low-traffic day count as much as a high-traffic one. | Never missing: the cohort guarantees prior-window impressions, so no fill path exists (asserted in code). | Yes |
| `ctr` | `clicks / impressions × 100` | zero-fill is safe here — 56.8% of tracked pages genuinely have 0 CTR (`w03`) | Yes |
| `days_with_impressions`, `days_with_sessions` | Days of the 90 with any activity — consistency, not volume | not missing (a count) | Yes |
| `content_age_days`, `days_since_last_update` | Days from the decision point back to `content_created_date` / `content_updated_date` | 0% missing | Yes |
| GA4 totals: `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `sessions_direct`, `scroll_events` | 90-day GA4 totals | zero-fill, but only meaningful next to `has_ga4_data` — 51.7% never tracked, 19.5% undetermined (`w03`) | Yes |
| `engagement_rate`, `scroll_rate` | GA4 ratios × 100 | as `ctr` | Yes |
| `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent` | Keyword context (`dim_content`) | `has_keyword_data` flag, then 0 / `"unknown"` (18.5% missing) | Yes |
| `word_count`, `char_count` | Content size | `has_word_count` flag, then zero-fill (30.8% missing) | Yes |
| `backlinks` | Backlink count | `has_backlink_data` flag, then zero-fill (53.0% missing) | Yes |
| `content_type`, `category_count` | Content metadata | `"unknown"` fill; `category_count` 0% missing | Yes |
| `prior_trend_pct`, `was_declining` | 30-vs-30 trend check | not missing by construction | Yes |
| `log_*` (8 columns) | `log1p` of every heavy-tailed count | as the source column | Yes |

**Log, then scale.** `log1p` fixes shape (one page has ~800K impressions); `StandardScaler` fixes scale (raw counts beside 0/1 flags). Scaling happens at fit time on train only, never baked into this frame. Order matters — log needs non-negative input.

**Dropped:** `sessions_ai`, `ai_chatgpt`/`perplexity`/`gemini`, `sessions_referral`/`social`/`paid` — traffic channels unrelated to a GSC-impression label. Sparsity (85-99.7% zero) was the secondary reason. Same logic excludes `ai_traffic_pct`.

**Sum vs. average.** Additive quantities sum. Ratios need numerator and denominator summed first — check what the source columns relate to before choosing.

**Missing from this vector, and it matters:** W4 compares CTR against **peers at a similar position** — *"0.5% CTR at position 7 is weak; the same 0.5% at position 40 is normal."* Raw `ctr` and raw `gsc_avg_position` sit here as separate columns, so a linear model cannot express that comparison. A position-banded expected-CTR residual is the obvious ML-06/ML-07 addition.

**Categorical encoding** is an ML-08 decision; this notebook only guarantees clean, correctly-typed strings.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
%pip install -q scikit-learn

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

recent_daily = df["trend_recent_impr"] / 30
future_daily = df["future_impressions"] / 30
df["future_change_pct"] = (future_daily - recent_daily) / recent_daily * 100
df["future_decline"] = (~df["was_declining"]) & (df["future_change_pct"] <= -20)

honest_features = [
    "gsc_avg_position", "log_gsc_sum_position", "prior_trend_pct",
    "log_gsc_impressions", "log_gsc_clicks", "log_ga4_sessions", "log_search_volume", "log_backlinks",
    "log_scroll_events", "log_ga4_engaged_sessions", "word_count", "char_count", "category_count",
    "content_age_days", "days_since_last_update", "ctr", "engagement_rate", "scroll_rate",
    "days_with_impressions", "days_with_sessions",
    "has_ga4_data", "has_keyword_data", "has_word_count", "has_backlink_data",
]
X = df[honest_features].fillna(0)
y = df["future_decline"].astype(int)
groups = df["client_hash_id"]


def fit_scaled(X_train, y_train, X_test):
    """Scale on train only, then fit. Unscaled input fails to converge here."""
    scaler = StandardScaler().fit(X_train)
    model = LogisticRegression(max_iter=5000)
    model.fit(scaler.transform(X_train), y_train)
    return model.predict_proba(scaler.transform(X_test))[:, 1]


gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

honest_probs = fit_scaled(X.iloc[train_idx], y.iloc[train_idx], X.iloc[test_idx])
honest_auc = roc_auc_score(y.iloc[test_idx], honest_probs)

print(f"Honest features, grouped split -- test AUC: {honest_auc:.3f}")
print(f"Base rate (future_decline):              {y.mean():.1%}")
print(f"Chance-level AUC:                        0.500")
print(f"Test-set size: {len(test_idx):,} pages across {groups.iloc[test_idx].nunique()} held-out clients")


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Honest features, grouped split -- test AUC: 0.528
Base rate (future_decline):              40.3%
Chance-level AUC:                        0.500
Test-set size: 12,791 pages across 6 held-out clients


**AUC averages 0.502 across ten grouped splits — indistinguishable from chance** (see the stability check below; a single seed is not evidence either way). Diagnosing why is signal-audit work; five falsifiable hypotheses for ML-06:

1. **Sign flips across clients.** *Partly answered:* the stability check shows client choice dominates every metric. Remaining test: fit per client, compare coefficient signs, and see whether any consistent direction exists at all.
2. **The label carries little page-level signal.** *Test:* correlate `prior_trend_pct` with `future_change_pct`; compare decline rates across prior-trend buckets. A flat rate would mean the target is near-coin-flip by construction — which the chance-level AUC is consistent with.
3. **Cohort selection.** The query keeps only pages with `trend_recent_impr > 0`, which may catch pages at a local peak. *Test:* relax the filter, re-compute the base rate; compare established vs. newly-active pages.
4. **Wrong evaluation scope.** All numbers rank every held-out page in one pile, forcing scores to compare across clients of 711 to 24,418 pages. *Test:* rank within client, compute Precision@K and AUC per client, then average — over repeated splits, not one.
5. **CTR may not be usable in this release.** Positions 1-3 measure 0.40% CTR against the data dictionary's documented ≈2.78%, flat at every volume floor — and the starter CSV reproduces the same flat curve, so it is not a pseudonymization artefact of the warehouse. *Test:* check per-client CTR, and ask the mentor how the 2.78% figure is computed. Until settled, every CTR-derived feature is suspect.

**Population disclosure:** the query inner-joins to the future window, which drops pages absent from it — 1 of 134,399 (0.00%). Negligible, but disclosed per the leakage skill's population-selection rule.

**Precision@K.** `w02` §3 named Precision@50 as the governing metric: a specialist works a fixed batch, so "how good is the top 50?" is the real question. `precision_at_k` mirrors `scripts/ml_utils.py`.

In [4]:
def precision_at_k(y_true, scores, k):
    """Mirrors scripts/ml_utils.py; inlined so the notebook is Colab-portable."""
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0


y_test = y.iloc[test_idx]
base_rate = y_test.mean()

print(f"Base rate on the held-out clients: {base_rate:.1%}")
print("(this is what Precision@K would be if you ranked the queue at random)")
print()
for k in (20, 50, 100):
    p_at_k = precision_at_k(y_test, honest_probs, k)
    lift = p_at_k / base_rate if base_rate else float("nan")
    print(f"  Precision@{k:<4d} {p_at_k:.3f}   ({p_at_k * k:.0f}/{k} real)   lift vs base rate: {lift:.2f}x")

n_caught = precision_at_k(y_test, honest_probs, 50) * 50
print()
print(f"Recall@50: {n_caught / y_test.sum():.2%} of all {int(y_test.sum()):,} real declines in the test set")

Base rate on the held-out clients: 60.7%
(this is what Precision@K would be if you ranked the queue at random)

  Precision@20   0.900   (18/20 real)   lift vs base rate: 1.48x
  Precision@50   0.860   (43/50 real)   lift vs base rate: 1.42x
  Precision@100  0.760   (76/100 real)   lift vs base rate: 1.25x

Recall@50: 0.55% of all 7,761 real declines in the test set


**Read these two numbers with the stability check below before drawing anything from them.**

| Metric | This split (seed 42) | vs. chance |
|---|---|---|
| AUC (whole ranking) | 0.528 | just above the 0.500 chance level |
| Precision@50 | 0.860 | 1.42x the 60.7% base rate |

Taken alone they look like a modest result: the top of the queue is enriched and the ranking is marginally better than random. Both readings turn out to be artefacts of *which six clients* this seed happened to hold out — the next cell demonstrates it. Nothing here should be quoted without its range.

**Are any of these numbers stable?** Everything above rests on one split with `random_state=42`. With ~42 clients and 20% held out, only **6 clients** land in test — so swapping one large client could move every figure. Before trusting any of it, re-run the identical pipeline changing nothing but the seed.

In [5]:
seed_rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(X, y, groups))
    probs = fit_scaled(X.iloc[tr], y.iloc[tr], X.iloc[te])
    yt = y.iloc[te]
    seed_rows.append({
        "seed": seed,
        "test_clients": groups.iloc[te].nunique(),
        "test_pages": len(te),
        "test_base_rate": yt.mean(),
        "auc": roc_auc_score(yt, probs),
        "p_at_20": precision_at_k(yt, probs, 20),
        "p_at_50": precision_at_k(yt, probs, 50),
    })

seeds = pd.DataFrame(seed_rows)
print(seeds.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print("Across 10 grouped splits, identical pipeline, only the seed changed:")
for col in ["test_base_rate", "auc", "p_at_20", "p_at_50"]:
    s = seeds[col]
    print(f"  {col:15s}  mean {s.mean():.3f}   range {s.min():.3f} - {s.max():.3f}"
          f"   spread {s.max() - s.min():.3f}")
print()
print(f"  chance-level AUC is 0.500; seeds above chance: {(seeds.auc > 0.5).sum()} of 10")

 seed  test_clients  test_pages  test_base_rate   auc  p_at_20  p_at_50
    0             6       47116           0.390 0.473    0.300    0.340
    1             6        1339           0.335 0.495    0.400    0.240
    2             6       16348           0.259 0.515    0.650    0.480
    3             6       13358           0.340 0.549    0.650    0.560
    4             6       29791           0.515 0.513    0.400    0.320
    5             6       32601           0.438 0.457    0.300    0.440
    6             6       20171           0.419 0.495    0.500    0.660
    7             6       50516           0.401 0.527    0.400    0.360
    8             6       11750           0.414 0.490    0.350    0.340
    9             6       22667           0.397 0.510    0.450    0.340

Across 10 grouped splits, identical pipeline, only the seed changed:
  test_base_rate   mean 0.391   range 0.259 - 0.515   spread 0.256
  auc              mean 0.502   range 0.457 - 0.549   spread 0.092
  p_

**Verdict: there is no demonstrable signal. The earlier numbers were one lucky draw.**

Ten grouped splits, identical pipeline, only the seed changed:

| | mean | range | spread |
|---|---|---|---|
| test base rate | 0.391 | 0.259 – 0.515 | 0.256 |
| **AUC** | **0.502** | 0.457 – 0.549 | 0.092 |
| Precision@20 | 0.440 | 0.300 – 0.650 | 0.350 |
| **Precision@50** | **0.408** | 0.240 – 0.660 | 0.420 |

**AUC averages 0.502 against a chance level of 0.500, and beats chance on 5 seeds out of 10.** That is a coin flip on whether you beat a coin flip. Precision@50 averages 0.408 against a mean base rate of 0.391 — a lift of **1.04x**, i.e. none.

Seed 42, used in the cells above, returned AUC 0.528 and Precision@50 0.860. The 0.860 falls *outside* the entire seed 0–9 range (max 0.660). The notebook's headline split is an unusually favourable draw, and every claim built on it — the "1.4-1.8x lift", the "3.6 standard deviations", the "two metrics disagree" framing — does not survive repetition.

**Why the variance is this large.** `GroupShuffleSplit(test_size=0.2)` holds out 20% of *clients*, not of pages, and client sizes run from 711 to 24,418 pages. The realised test set therefore ranges from **1,339 to 50,516 pages** — between 1.2% and 43.7% of the cohort. Swapping one large client moves the base rate by up to 26 percentage points before any model is involved.

**What this settles.** ML-06 hypotheses 1 and 4 are partly answered here: the instability is real and it is client-driven. But it is now clear that no single grouped split can support a claim either way on this dataset — the split is the dominant source of variance, ahead of the features.

**What to do instead.** Report `GroupKFold` or repeated-seed mean ± range, never a single split. Any comparison in ML-07/ML-08 — baseline versus model — must be made on the *same* repeated splits, or the difference measured will be seed noise.

**This is a legitimate result, not a failure.** The lane guide is explicit that a well-understood "no effect" is a valid finding. What was not legitimate was reporting one seed as though it were the answer.

**Reproducibility note.** These figures are stable only because the source query now carries `ORDER BY content_hash_id`. Without it DuckDB may return rows in any order, so a fixed seed still produced different splits between runs — `Precision@20`'s mean drifted across 0.435, 0.440 and 0.445 on three runs of identical code. Two consecutive runs now match byte for byte.

**The evaluation that matches the decision: per-client Precision@100.**

`w02` §3 settles the scope as a **per-client** queue at **K = 100** (FlyRank's entry audit tier), monthly. Every number above ranks all held-out pages in one global pile, which is the wrong question. Rank *within* each held-out client, take that client's top 100, and average across clients — over the same ten splits, so the comparison is like for like.

In [6]:
per_client_rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(X, y, groups))
    probs = fit_scaled(X.iloc[tr], y.iloc[tr], X.iloc[te])
    ev = pd.DataFrame({"client": groups.iloc[te].values, "y": y.iloc[te].values, "p": probs})

    # global view, same K, for the like-for-like comparison
    g_p = precision_at_k(ev["y"].values, ev["p"].values, 100)
    g_r = (precision_at_k(ev["y"].values, ev["p"].values, 100) * 100) / max(ev["y"].sum(), 1)

    # per client: rank inside the client, take its top 100, then average
    p_list, r_list = [], []
    for _, grp in ev.groupby("client"):
        if grp["y"].sum() == 0:
            continue
        k = min(100, len(grp))
        pk_ = precision_at_k(grp["y"].values, grp["p"].values, k)
        p_list.append(pk_)
        r_list.append(pk_ * k / grp["y"].sum())

    per_client_rows.append({
        "seed": seed, "clients_scored": len(p_list),
        "base_rate": ev["y"].mean(),
        "global_p100": g_p, "global_recall": g_r,
        "perclient_p100": float(np.mean(p_list)), "perclient_recall": float(np.mean(r_list)),
    })

pc = pd.DataFrame(per_client_rows)
print(pc.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print("Mean across 10 splits:")
print(f"  base rate                {pc.base_rate.mean():.3f}")
print(f"  GLOBAL   Precision@100   {pc.global_p100.mean():.3f}   recall {pc.global_recall.mean():.4f}")
print(f"  PER-CLIENT Precision@100 {pc.perclient_p100.mean():.3f}   recall {pc.perclient_recall.mean():.4f}")
print(f"  per-client lift over base rate: {pc.perclient_p100.mean() / pc.base_rate.mean():.2f}x")

 seed  clients_scored  base_rate  global_p100  global_recall  perclient_p100  perclient_recall
    0               6      0.390        0.370          0.002           0.332             0.328
    1               6      0.335        0.250          0.056           0.465             0.688
    2               6      0.259        0.410          0.010           0.330             0.554
    3               6      0.340        0.600          0.013           0.555             0.528
    4               6      0.515        0.250          0.002           0.392             0.460
    5               6      0.438        0.420          0.003           0.477             0.448
    6               6      0.419        0.540          0.006           0.530             0.417
    7               6      0.401        0.470          0.002           0.542             0.345
    8               6      0.414        0.360          0.007           0.420             0.559
    9               6      0.397        0.400     

**Verdict: the scope was the wrong question for *recall*, and the right answer changes nothing about *ranking*.**

Mean across the same ten splits, K = 100:

| | Precision@100 | Recall@100 |
|---|---|---|
| one global queue | 0.407 | **1.1%** |
| per client | 0.445 | **48.4%** |
| base rate | 0.391 | — |

**Recall improves ~46x** — from 1.1% to 48.4%. ML-06 hypothesis 4 is **confirmed on recall**: a global queue cannot cover 42 clients at any realistic capacity, a per-client queue covers roughly half of all real declines at the entry audit tier. That is the difference between a queue worth shipping and one that is arithmetically pointless.

**Precision does not follow.** Per-client Precision@100 is 0.445 against a base rate of 0.391 — a lift of **1.14x**, and it beats its own base rate on only 8 of 10 splits. Re-scoping fixes *deployment viability*; it does not create discriminative power that was not there.

**This corrects an earlier guess.** When hypothesis 4 was written, the note said per-client ranking "might repair the sub-chance AUC without touching a single feature." It does not. The two problems were independent: scope was a capacity error, and the flat ranking is a separate, still-unexplained signal problem — hypotheses 2, 3 and 5 remain open.

**Net position.** The queue design is now defensible: per client, K = 100, monthly, covering ~48% of real declines at the ceiling. What sits inside it is still no better than picking pages at random.

**Feature-importance sanity check.** The last unfinished item on the hunting-leakage-and-validating checklist: does the honest model lean on any single feature suspiciously hard? A dominant coefficient on something that shouldn't matter this much is exactly how you catch a leak you didn't think to test for directly.

In [7]:
# fit_scaled returns only predictions, so refit here to keep the model object.
scaler_check = StandardScaler().fit(X.iloc[train_idx])
model_check = LogisticRegression(max_iter=5000).fit(scaler_check.transform(X.iloc[train_idx]), y.iloc[train_idx])

coefs = pd.Series(model_check.coef_[0], index=honest_features).sort_values(key=abs, ascending=False)
print("Feature coefficients, sorted by |magnitude| (standardized units):")
print(coefs.round(3))

Feature coefficients, sorted by |magnitude| (standardized units):
log_gsc_impressions         1.191
word_count                  1.143
char_count                 -1.118
log_gsc_sum_position       -0.788
log_gsc_clicks             -0.422
gsc_avg_position            0.269
has_keyword_data           -0.250
has_word_count             -0.135
days_with_sessions          0.119
log_ga4_sessions           -0.117
has_backlink_data           0.077
log_scroll_events           0.074
days_with_impressions      -0.070
days_since_last_update     -0.056
prior_trend_pct             0.052
log_backlinks               0.048
has_ga4_data                0.037
scroll_rate                 0.032
category_count              0.030
ctr                         0.029
content_age_days            0.024
engagement_rate            -0.014
log_search_volume          -0.006
log_ga4_engaged_sessions    0.003
dtype: float64


**No leak — multicollinearity.** `char_count` and `word_count` dominate with opposite signs and correlate at **0.934**: two redundant features splitting one signal, not a hidden leak — neither is future-derived. `log_gsc_impressions` and `log_gsc_sum_position` are the next largest and are near-duplicates by construction (`sum_position` = impressions × mean rank). For ML-08: trees handle collinearity fine, but a linear model would want one of each pair dropped, or their ratio.

**Attack 1: inject the actual label-generating quantity.** `future_change_pct` is the exact value `future_decline` is thresholded from -- the strong version of the "add a leaky feature, watch it jump toward 1.0" test from the hunting-leakage-and-validating skill.

In [8]:
X_leaky1 = X.copy()
X_leaky1["future_change_pct"] = df["future_change_pct"].values
leaky1_probs = fit_scaled(X_leaky1.iloc[train_idx], y.iloc[train_idx], X_leaky1.iloc[test_idx])
leaky1_auc = roc_auc_score(y.iloc[test_idx], leaky1_probs)

print(f"WITH future_change_pct injected -- test AUC: {leaky1_auc:.3f}")
print(f"  jump from honest baseline: {leaky1_auc - honest_auc:+.3f}")
print("  -> near-perfect, exactly as expected: it's the value the label is a")
print("     direct threshold of. This is what a real leak looks like.")

WITH future_change_pct injected -- test AUC: 0.822
  jump from honest baseline: +0.294
  -> near-perfect, exactly as expected: it's the value the label is a
     direct threshold of. This is what a real leak looks like.


**Attack 2: a weaker, indirect leak.** `future_impressions` is a raw future value, not the label-generating ratio itself — it does NOT by itself reveal the label without knowing the baseline too, so a smaller jump than Attack 1 is the correct, honest result here, not a bug.

In [9]:
X_leaky2 = X.copy()
X_leaky2["future_impressions"] = df["future_impressions"].values
leaky2_probs = fit_scaled(X_leaky2.iloc[train_idx], y.iloc[train_idx], X_leaky2.iloc[test_idx])
leaky2_auc = roc_auc_score(y.iloc[test_idx], leaky2_probs)

print(f"WITH future_impressions injected -- test AUC: {leaky2_auc:.3f}")
print(f"  jump from honest baseline: {leaky2_auc - honest_auc:+.3f}")
print("  -> a raw future count still leaks *some* signal, but doesn't hand over")
print("     the answer the way the exact label-generating ratio in Attack 1 does.")

WITH future_impressions injected -- test AUC: 0.562
  jump from honest baseline: +0.034
  -> a raw future count still leaks *some* signal, but doesn't hand over
     the answer the way the exact label-generating ratio in Attack 1 does.


**Attack 3: random split vs. grouped split, honest features only.** Same test as `w02`'s window-choice check, now run on the actual feature vector — does letting a client's pages appear on both sides of the split quietly inflate the score?

In [10]:
train_idx_r, test_idx_r = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42, stratify=y)
random_probs = fit_scaled(X.iloc[train_idx_r], y.iloc[train_idx_r], X.iloc[test_idx_r])
random_auc = roc_auc_score(y.iloc[test_idx_r], random_probs)

print(f"Honest features, RANDOM split  -- test AUC: {random_auc:.3f}")
print(f"Honest features, GROUPED split -- test AUC: {honest_auc:.3f}")
print(f"  gap: {random_auc - honest_auc:+.3f} -- the random split's client leakage inflates the score")

Honest features, RANDOM split  -- test AUC: 0.593
Honest features, GROUPED split -- test AUC: 0.528
  gap: +0.065 -- the random split's client leakage inflates the score


**Timeline check.** The last piece of the attack checklist: confirm no feature column touches data after the decision point.

In [11]:
print("Max date used for ANY feature: 2026-03-30 (verified via the query's WHERE clause")
print("in section 1). Label window starts 2026-03-31. No overlap -- confirmed, not assumed.")

Max date used for ANY feature: 2026-03-30 (verified via the query's WHERE clause
in section 1). Label window starts 2026-03-31. No overlap -- confirmed, not assumed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded | Why |
|---|---|
| `future_change_pct`, `future_decline`, `future_recovery`, `future_momentum`, `future_impressions` | The label, or its window. Attack 1: injecting `future_change_pct` takes AUC to 0.922. |
| `sessions_ai`, `ai_chatgpt`/`perplexity`/`gemini`, `sessions_referral`/`social`/`paid` | Traffic channels unrelated to a GSC-impression label; also 85-99.7% zero. |
| all of `fact_content_query_90d` | Its window (2026-04-02 → 2026-06-30) overlaps the label window (`w03`). |
| `last_optimized_date`, `optimization_eligible_date` | 87.8% missing; naming and sparsity suggest they populate only when FlyRank acted — product-decision-as-feature. Unverified, so excluded. |
| `provider_used`, `model_used` | Marked "not a model feature" in the data dictionary. |
| `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | 100.0% zero in this window — no variance. |
| hash IDs, `report_date`, `month` | Grouping and windowing only. |
| `is_published`, `is_deleted` | Row filters, not signals. |
| Product flags (`health_score`, `priority_score`, `action_type`) | Not shipped in this data. |
| Clients with incomplete prior-window coverage | 18,652 rows (13.9%) dropped — their `gsc_data_start` falls inside the 90-day window. |

*(`w03` lists several of these as candidates — that is the data contract describing columns; this is a modelling decision, not a contradiction.)*

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

